In [3]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import re
from datetime import datetime
import openpyxl
import logging
import shutil
import tempfile
from PyPDF2 import PdfReader
import camelot as cam
from collections import defaultdict

# Configure logging and start timer
start = time.time()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
pd.set_option('display.max_rows', 500, 'display.max_columns', 500, 'display.max_colwidth', 1000)

# Setup directories
BASE_DIR = "cre_financial_data"
ORIGINAL_DIR = os.path.join(BASE_DIR, "original")
CURRENT_DIR = os.path.join(BASE_DIR, "current")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Create subdirectories for categorized files
for subfolder in ['balance-sheet', 'income-statement']:
    os.makedirs(os.path.join(CURRENT_DIR, subfolder), exist_ok=True)
    print(f"📁 Created directory: {os.path.join(CURRENT_DIR, subfolder)}")

for dir_path in [BASE_DIR, ORIGINAL_DIR, OUTPUT_DIR]:
    os.makedirs(dir_path, exist_ok=True)
    print(f"📁 Created directory: {dir_path}")

# Log file for tracking processed files
PROCESSED_FILES_LOG = os.path.join(BASE_DIR, "processed_files.csv")

bank_url = 'https://bankacredins-ks.com/kush-jemi/raporte-dhe-njoftime/raporte-vjetore/'
print(f"🔗 Target URL: {bank_url}")

def load_processed_files():
    """Load list of already processed files from CSV"""
    if os.path.exists(PROCESSED_FILES_LOG):
        df = pd.read_csv(PROCESSED_FILES_LOG)
        return set(df['file_name'].tolist())
    return set()

def save_processed_file(file_name, status=1, reporting_date=None, current_name=None, is_corrupt=0):
    """Save processed file to log"""
    processed = load_processed_files()
    if file_name not in processed:
        df = pd.DataFrame({
            'file_name': [file_name],
            'processed_date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            'status': [status],
            'reporting_date': [reporting_date],
            'current_name': [current_name],
            'is_corrupt': [is_corrupt]
        })
        if os.path.exists(PROCESSED_FILES_LOG):
            existing = pd.read_csv(PROCESSED_FILES_LOG)
            df = pd.concat([existing, df], ignore_index=True)
        df.to_csv(PROCESSED_FILES_LOG, index=False)

def scrape_pdf_links(url):
    """Scrape PDF and Excel links from the Credins page"""
    print(f"🔍 Attempting to connect to: {url}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        print(f"✅ Successfully connected! Status code: {response.status_code}")
        
        soup = BeautifulSoup(response.text, 'html.parser')
        file_links = []
        
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.lower().endswith('.xlsx') or (href.lower().endswith('.pdf') and 'web' in href.lower()):
                full_url = urljoin(url, href)
                file_links.append(full_url)
                print(f"   📄 Found file: {os.path.basename(full_url)}")
        
        print(f"\n✅ Total files found: {len(file_links)}")
        return file_links
    except requests.exceptions.RequestException as e:
        print(f"❌ Error connecting to website: {e}")
        return []

def download_file(url, local_dir):
    """Download file (PDF or Excel) to local directory"""
    file_name = os.path.basename(url)
    local_path = os.path.join(local_dir, file_name)
    
    print(f"   ⬇️ Downloading: {file_name}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open(local_path, 'wb') as f:
            f.write(response.content)
        
        file_size = os.path.getsize(local_path) / 1024
        print(f"   ✅ Downloaded: {file_name} ({file_size:.1f} KB)")
        return local_path
    except Exception as e:
        print(f"   ❌ Failed to download {file_name}: {e}")
        return None

def extract_date_from_filename(filename):
    """Extract date from filename like 31.12.2023"""
    match = re.search(r'\d{2}\.\d{2}\.\d{4}', filename)
    if match:
        return match.group()
    return None

def get_file_creation_date(file_path, file_extension):
    """Extract creation date from file metadata"""
    dt = None
    try:
        if file_extension == 'xlsx':
            wb = openpyxl.load_workbook(file_path)
            properties = wb.properties
            created = properties.created
            modified = properties.modified
            created_date = created.date() if created else None
            modified_date = modified.date() if modified else None
            dt = created_date or modified_date
        elif file_extension == 'pdf':
            reader = PdfReader(file_path)
            metadata = reader.metadata
            
            creation_date_raw = metadata.get('/CreationDate', None)
            modified_date_raw = metadata.get('/ModDate', None)
            
            if modified_date_raw and modified_date_raw.startswith("D:"):
                modified_date_raw = modified_date_raw[2:].split('+')[0].split('-')[0]
                modified_date = datetime.strptime(modified_date_raw, "%Y%m%d%H%M%S")
                dt = modified_date
            elif creation_date_raw and creation_date_raw.startswith("D:"):
                creation_date_raw = creation_date_raw[2:].split('+')[0].split('-')[0]
                creation_date = datetime.strptime(creation_date_raw, "%Y%m%d%H%M%S")
                dt = creation_date
    except Exception as e:
        print(f"   ⚠️ Could not read metadata: {e}")
    
    return dt

def download_all(force=False):
    """Download files and categorize them locally"""
    processed = set() if force else load_processed_files()
    if force:
        print("⚠️ Force mode enabled: all files will be downloaded")
    
    print(f"🔍 Scraping file links from: {bank_url}")
    file_links = scrape_pdf_links(bank_url)
    
    if not file_links:
        print("❌ No files found")
        return []
    
    new_files = []
    file_metadata = []

    print(f"\n📥 Starting download of {len(file_links)} files...")
    
    for i, file_url in enumerate(file_links, 1):
        file_name = os.path.basename(file_url)
        print(f"\n[{i}/{len(file_links)}] Processing: {file_name}")
        
        # Skip if already processed
        if file_name in processed and not force:
            print(f"   ⏭️ File already processed. Skipping...")
            continue

        new_files.append(file_name)
        
        # Download to original directory
        file_path = download_file(file_url, ORIGINAL_DIR)
        if not file_path:
            continue

        file_extension = file_name.split('.')[-1].lower()
        
        # Get file creation date from metadata
        dt = get_file_creation_date(file_path, file_extension)
        if dt:
            print(f"   📅 File creation date: {dt}")

        # Extract date from filename
        date = extract_date_from_filename(file_name)
        
        if date:
            print(f"   📅 Extracted reporting date: {date}")
        else:
            print(f"   ⚠️ Could not extract date from filename")
            date = datetime.now().strftime('%d.%m.%Y')

        # For Credins, we create both BS and IS from the same file
        if file_extension == 'xlsx':
            bs_new_name = f'cre_bs_{date}.xlsx'
            is_new_name = f'cre_is_{date}.xlsx'
        else:  # pdf
            bs_new_name = f'cre_bs_{date}.pdf'
            is_new_name = f'cre_is_{date}.pdf'
        
        # Copy to both categorized folders
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'balance-sheet', bs_new_name))
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'income-statement', is_new_name))
        
        file_metadata.append({
            'file_name': file_name,
            'file_path': file_path,
            'bs_current_name': bs_new_name,
            'is_current_name': is_new_name,
            'file_extension': file_extension,
            'reporting_date': date,
            'creation_date': dt,
            'download_date': datetime.now(),
            'is_corrupt': 0
        })
        
        print(f"   ✅ Created: {bs_new_name} and {is_new_name}")
        
        # Mark as processed
        save_processed_file(file_name, reporting_date=date, current_name=f"{bs_new_name}|{is_new_name}")

    # Save metadata
    if file_metadata:
        metadata_df = pd.DataFrame(file_metadata)
        metadata_path = os.path.join(OUTPUT_DIR, f"file_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        metadata_df.to_csv(metadata_path, index=False)
        print(f"\n💾 File metadata saved to {metadata_path}")

    if new_files:
        print(f"\n✅ Downloaded {len(new_files)} new files.")
    else:
        print("\n✅ No new files detected.")

    # Return list of all new current files
    current_files = []
    for m in file_metadata:
        current_files.append(m['bs_current_name'])
        current_files.append(m['is_current_name'])
    return current_files

def replace_negatives(value):
    """Convert parenthesized numbers to negative"""
    if pd.isna(value) or str(value).strip() in ['', '-', '0']:
        return '0'
    val = str(value).strip()
    if '(' in val and ')' in val:
        num = val.replace('(', '').replace(')', '').replace(',', '').strip()
        return f"-{num}"
    return val

def read_cre_pdf(pdf_path, pdf_filename):
    """Read Credins PDF file (balance sheet page 1, income statement page 3)"""
    try:
        print(f"      📊 Reading PDF: {pdf_filename}")
        
        # Balance sheet from page 1 (stream)
        bal_tables = cam.read_pdf(pdf_path, pages='1', flavor='stream')
        # Income statement from page 3 (lattice)
        inc_tables = cam.read_pdf(pdf_path, pages='3', flavor='lattice')

        bal_dfs = []
        inc_dfs = []

        for table in bal_tables:
            bal_dfs.append(table.df)
        
        for table in inc_tables:
            inc_dfs.append(table.df)
        
        full_bal = pd.concat(bal_dfs, ignore_index=True)
        full_inc = pd.concat(inc_dfs, ignore_index=True)

        # Clean balance sheet
        full_bal = full_bal[(full_bal[0] != '') & (full_bal[1] != '') & (full_bal[2] != '')]
        full_bal[0] = full_bal[0].replace('/ .*', '', regex=True)
        full_bal[1] = pd.to_numeric(full_bal[1].astype(str).str.replace('-', '0').str.replace(',', '').apply(replace_negatives))
        full_bal[2] = pd.to_numeric(full_bal[2].astype(str).str.replace('-', '0').str.replace(',', '').apply(replace_negatives))
        full_bal.fillna(0, inplace=True)

        # Clean income statement
        full_inc.drop(columns=[1], inplace=True)
        full_inc = full_inc[(full_inc[0] != '') & (full_inc[2] != '') & (full_inc[3] != '')]
        full_inc[0] = full_inc[0].replace('/ .*', '', regex=True).replace('\n.*', '', regex=True)
        full_inc[2] = pd.to_numeric(full_inc[2].astype(str).str.replace('-', '0').str.replace(',', '').apply(replace_negatives))
        full_inc[3] = pd.to_numeric(full_inc[3].astype(str).str.replace('-', '0').str.replace(',', '').apply(replace_negatives))
        full_inc.fillna(0, inplace=True)

        full_bal.columns = ['BALANCE_CATEGORY', 'CURRENT_BALANCE_VALUE', 'PREVIOUS_BALANCE_VALUE']
        full_inc.columns = ['INCOME_CATEGORY', 'CURRENT_INCOME_VALUE', 'PREVIOUS_INCOME_VALUE']

        # Extract date from filename
        date = extract_date_from_filename(pdf_filename)
        if date:
            date = date.replace('.', '/')
        
        full_bal['DT_REPORT'] = date
        full_inc['DT_REPORT'] = date
        full_bal['FILE_NAME'] = pdf_filename
        full_inc['FILE_NAME'] = pdf_filename.replace('bs', 'is')

        print(f"      ✅ Extracted: {len(full_bal)} balance rows, {len(full_inc)} income rows")
        return full_bal, full_inc
    
    except Exception as e:
        print(f"      ❌ Error reading PDF: {e}")
        return pd.DataFrame(), pd.DataFrame()

def read_cre_excel(excel_path, excel_filename):
    """Read Credins Excel file (BS and IS sheets)"""
    try:
        print(f"      📊 Reading Excel: {excel_filename}")
        
        # Read balance sheet sheet (default)
        df = pd.read_excel(excel_path)
        df = df.dropna(subset=['Unnamed: 1'])
        df.drop(['Unnamed: 0', 'Unnamed: 2'], axis=1, inplace=True)
        
        # Handle column rearrangement if needed
        try:
            df['Unnamed: 4'] = df['Unnamed: 5']
            df.drop(['Unnamed: 5'], axis=1, inplace=True)
        except:
            pass
        
        # Clean numeric values
        df['Unnamed: 3'] = df['Unnamed: 3'].apply(lambda x: round(x, 0))
        df['Unnamed: 4'] = df['Unnamed: 4'].apply(lambda x: round(x, 0))
        df['Unnamed: 3'] = pd.to_numeric(df['Unnamed: 3'])
        df['Unnamed: 4'] = pd.to_numeric(df['Unnamed: 4'])
        df = df.dropna(subset=['Unnamed: 3', 'Unnamed: 4'], how='all')
        df.fillna(0, inplace=True)
        
        # Read income statement sheet
        df2 = pd.read_excel(excel_path, sheet_name='IS')
        df2 = df2.dropna(subset=['Unnamed: 1'])
        df2.drop(['Unnamed: 0', 'Unnamed: 2'], axis=1, inplace=True)
        
        # Handle column rearrangement if needed
        try:
            df2['Unnamed: 4'] = df2['Unnamed: 5']
            df2.drop(['Unnamed: 5'], axis=1, inplace=True)
        except:
            pass
        
        # Clean numeric values
        df2['Unnamed: 3'] = df2['Unnamed: 3'].apply(lambda x: round(x, 0))
        df2['Unnamed: 4'] = df2['Unnamed: 4'].apply(lambda x: round(x, 0))
        df2['Unnamed: 3'] = pd.to_numeric(df2['Unnamed: 3'])
        df2['Unnamed: 4'] = pd.to_numeric(df2['Unnamed: 4'])
        df2 = df2.dropna(subset=['Unnamed: 3', 'Unnamed: 4'], how='all')
        df2.fillna(0, inplace=True)
        
        # Set column names
        df.columns = ['BALANCE_CATEGORY', 'CURRENT_BALANCE_VALUE', 'PREVIOUS_BALANCE_VALUE']
        df2.columns = ['INCOME_CATEGORY', 'CURRENT_INCOME_VALUE', 'PREVIOUS_INCOME_VALUE']
        
        # Extract date from filename
        date = extract_date_from_filename(excel_filename)
        if date:
            date = date.replace('.', '/')
        
        df['DT_REPORT'] = date
        df2['DT_REPORT'] = date
        df['FILE_NAME'] = excel_filename
        df2['FILE_NAME'] = excel_filename.replace('bs', 'is')
        
        print(f"      ✅ Extracted: {len(df)} balance rows, {len(df2)} income rows")
        return df, df2
    
    except Exception as e:
        print(f"      ❌ Error reading Excel: {e}")
        return pd.DataFrame(), pd.DataFrame()

# Data storage
income_data = defaultdict(list)
balance_data = defaultdict(list)

def process_cre_files():
    """Process all Credins files and populate data dictionaries"""
    
    # Process balance sheet files
    bs_dir = os.path.join(CURRENT_DIR, 'balance-sheet')
    if os.path.exists(bs_dir):
        print(f"\n📂 Processing balance sheet files...")
        for filename in os.listdir(bs_dir):
            if filename.startswith('cre_bs_'):
                file_path = os.path.join(bs_dir, filename)
                print(f"\n   🔍 Processing: {filename}")
                
                # Extract date from filename
                date_match = re.search(r'\d{2}\.\d{2}\.\d{4}', filename)
                dt_report = date_match.group() if date_match else None
                dt_report_formatted = datetime.strptime(dt_report, '%d.%m.%Y').strftime('%Y-%m-%d') if dt_report else None
                
                if not dt_report:
                    print(f"      ⚠️ Could not extract date from filename")
                    continue
                
                file_extension = filename.split('.')[-1].lower()
                
                # Get corresponding income statement filename
                is_filename = filename.replace('bs', 'is')
                is_path = os.path.join(CURRENT_DIR, 'income-statement', is_filename)
                
                if os.path.exists(is_path):
                    # Read based on file type
                    if file_extension == 'pdf':
                        balance_df, income_df = read_cre_pdf(file_path, filename)
                    else:  # xlsx
                        balance_df, income_df = read_cre_excel(file_path, filename)
                    
                    # Add to balance data
                    if not balance_df.empty and dt_report_formatted:
                        for _, row in balance_df.iterrows():
                            balance_data[dt_report_formatted].append({
                                'Category': row['BALANCE_CATEGORY'],
                                'PREVIOUS_QUARTER': row['PREVIOUS_BALANCE_VALUE'],
                                'CURRENT_QUARTER': row['CURRENT_BALANCE_VALUE'],
                                'DT_REPORT': dt_report_formatted,
                                'FILE_NAME': filename
                            })
                        print(f"      ✅ Added {len(balance_df)} balance rows for {dt_report_formatted}")
                    
                    # Add to income data
                    if not income_df.empty and dt_report_formatted:
                        for _, row in income_df.iterrows():
                            income_data[dt_report_formatted].append({
                                'Category': row['INCOME_CATEGORY'],
                                'PREVIOUS_QUARTER': row['PREVIOUS_INCOME_VALUE'],
                                'CURRENT_QUARTER': row['CURRENT_INCOME_VALUE'],
                                'DT_REPORT': dt_report_formatted,
                                'FILE_NAME': is_filename
                            })
                        print(f"      ✅ Added {len(income_df)} income rows for {dt_report_formatted}")

def clean_numeric_values(df, prev_col, curr_col):
    """Clean and convert numeric columns"""
    for col in [prev_col, curr_col]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

def main(force=False, extract_only=False):
    """Main execution function that returns unified DataFrames"""
    
    print("\n" + "="*60)
    print("🏦 CREDINS BANK FINANCIAL DATA EXTRACTION")
    print("="*60)
    
    # Clear previous data
    global income_data, balance_data
    income_data.clear()
    balance_data.clear()
    
    # Decide which files to process
    if extract_only:
        print("\n📂 EXTRACT-ONLY MODE: Processing existing files")
        new_files = []
        for category in ['balance-sheet', 'income-statement']:
            category_dir = os.path.join(CURRENT_DIR, category)
            if os.path.exists(category_dir):
                category_files = [f for f in os.listdir(category_dir) if f.startswith('cre_') and f.endswith(('.pdf', '.xlsx'))]
                new_files.extend(category_files)
        print(f"Found {len(new_files)} existing files")
    else:
        print("\n🌐 DOWNLOAD MODE: Downloading and processing new files")
        new_files = download_all(force=force)
    
    if new_files or extract_only:
        # Process all Credins files
        process_cre_files()
        
        # Create DataFrames
        print("\n📊 Creating DataFrames...")
        
        # Balance Sheet DataFrame
        full_bs = pd.DataFrame()
        for date, data in balance_data.items():
            df = pd.DataFrame(data)
            full_bs = pd.concat([full_bs, df], ignore_index=True)
        
        if not full_bs.empty:
            full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_bs = clean_numeric_values(full_bs, 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE')
            
            # Category mapping for balance sheet
            bs_map = {
                'Paraja e gatshme dhe gjendja me BQK-në': '20',
                'Paraja e gatshme dhe gjendja me BQK-në/ Cash and balances with CBK/ Bilans gotovine sa CBK': '20',
                'Kërkesat ndaj bankave': '21',
                'Kërkesat ndaj bankave/ Claims on banks/ Potraživanja bankama': '21',
                'Bonot e thesarit': '22',
                'Investimet në letra me vlerë': '23',
                'Investimet në letra me vlerë/ Investment Securities/ Hartije od vrednosti': '23',
                'Kreditë dhe paradhëniet ndaj klientëve': '26',
                'Kreditë dhe paradhëniet ndaj klientëve/ Loans and advances to customers/ Plasmani klijentima': '26',
                'Patundshmëritë dhe pajisjet': '27',
                'Patundshmëritë dhe pajisjet/ Property and equipment/ Imovina I oprema': '27',
                'Pasuritë e paprekshme': '28',
                'Pasuritë e paprekshme/ Intangible assets/ Nematerijalna aktiva': '28',
                'Pasuritë tatimore të shtyra': '29',
                'Pasuritë tatimore të shtyra/ Deferred tax assets/ Odložena poreska sredstva': '29',
                'Pasuritë tjera': '30',
                'Pasuritë tjera/ Other assets/ Ostala aktiva': '30',
                'Gjithsej pasuritë': '31',
                'Gjithsej pasuritë/ Total assets/ Ukupna aktiva': '31',
                'Depozitat e klientëve': '32',
                'Depozitat e klientëve/ Customer deposits/ Depoziti klijenata': '32',
                'Detyrimet ndaj bankave': '33',
                'Detyrimet ndaj bankave/ Due to banks/ Obaveze prema banci': '33',
                'Fondet tjera të huazuara': '34',
                'Detyrimet tatimore të shtyra': '35',
                'Detyrimet tatimore të shtyra/ Deferred tax liabilities/ Odlozene poreske obaveze': '35',
                'Detyrimet tjera': '36',
                'Detyrimet tjera/ Other liabilities/ Druge obaveze ': '36',
                'Gjithsej detyrimet': '37',
                'Gjithsej detyrimet/ Total liabilities/ Ukupne obaveze ': '37',
                'Kapitali aksionar': '38',
                'Kapitali aksionar/ Share capital/ Deoničarske akcize': '38',
                'Primi I aksioneve': '39',
                'Primi I aksioneve/ Share premium/ Akcionarska premija': '39',
                'Rezervat e kapitalit': '40',
                'Fitimi i mbajtur/(humbja) nga vitet paraprake': '41',
                'Fitimi i mbajtur/(humbja) nga vitet paraprake/ Retained profit/loss from previous years/ Nerasporedena dobit/gubitak iz prethodnih godina': '41',
                'Fitimi/(humbja) e vitit aktual': '42',
                'Fitimi/(humbja) e vitit aktual/ current year profit/loss/ gubitak iz prethodnih godina': '42',
                'Përbërësit tjerë të ekuitetit': '43',
                'Përbërësit tjerë të ekuitetit/ Other equity capital components/ Ostale komponente akcijskog kapitala ': '43',
                'Gjithsej ekuiteti i aksionarëve': '44',
                'Gjithsej ekuiteti i aksionarëve/ Total shareholder\'s equity/ Ukupni kapital deoničara': '44',
                'Gjithsej detyrimet dhe ekuiteti i aksionarëve': '45',
                'Gjithsej detyrimet dhe ekuiteti i aksionarëve/ Total liabilities and shareholder\'s equity/ Ukupne obaveze i kapital deoničara': '45',
            }
            
            full_bs['CATEGORY_CODE'] = full_bs['BALANCE_CATEGORY'].map(bs_map)
            full_bs['BANK_ID'] = 11  # Credins Bank ID
            full_bs['STATEMENT_TYPE'] = 'BALANCE_SHEET'
            full_bs['DT_REPORT'] = pd.to_datetime(full_bs['DT_REPORT'], format='%d/%m/%Y', errors='coerce')
        
        # Income Statement DataFrame
        full_is = pd.DataFrame()
        for date, data in income_data.items():
            df = pd.DataFrame(data)
            full_is = pd.concat([full_is, df], ignore_index=True)
        
        if not full_is.empty:
            full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_is = clean_numeric_values(full_is, 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE')
            
            # Category mapping for income statement
            is_map = {
                'Të hyrat nga interesi': '1',
                'Të hyrat nga interesi/ interest income/ kamatni prihod': '1',
                'Shpenzimet e interesit': '2',
                'Shpenzimet e interesit/ interest expense/ kamatni rashod': '2',
                'Neto të hyrat nga interesi': '3',
                'Neto të hyrat nga interesi/ net interest income/ neto kamatni prihod': '3',
                'Të hyrat nga tarifat dhe komisionet': '4',
                'Të hyrat nga tarifat dhe komisionet/ Fee and commission income/ Prihod od naknada i provizija': '4',
                'Shpenzimet e tarifave dhe komisioneve': '5',
                'Shpenzimet e tarifave dhe komisioneve/ Fee and commission expense/ Rashod od naknada i provizija': '5',
                'Neto të hyrat nga tarifat dhe komisionet': '6',
                'Neto të hyrat nga tarifat dhe komisionet/ Net fee and commission income/ Neto dobitak na ime naknada i provizija': '6',
                'Neto të hyrat nga tregtimi': '7',
                'Neto të hyrat nga tregtimi/ Net trading profit/ Neto profit od trgovanja': '7',
                'Neto të hyrat nga instrumentet tjera financiare': '8',
                'Neto të hyrat nga instrumentet tjera financiare/ Net income from other financial instruments/ Neto prihod iz drugih finansijskih instrumenata': '8',
                'Neto të hyrat (shpenzimet) tjera operative': '9',
                'Neto të hyrat (shpenzimet) tjera operative/ Net other operating income (expense) / Drugi neto operativni prihod (rashod)': '9',
                'Gjithsej të hyrat': '10',
                'Gjithsej të hyrat/ Total income / Ukupna dobit': '10',
                'Provizionet për humbjet nga kreditë': '11',
                'Provizionet për humbjet nga kreditë/ Impairment losses on loans/ Gubici na problematičnim kreditima': '11',
                'Provizionet tjera': '13',
                'Fitimi/(humbja) para tatimit': '14',
                'Fitimi/(humbja) para tatimit/ Profit/(loss) before taxation/Profit / (gubitak) pre oporezivanja': '14',
                'Shpenzimet e tatimit në fitim': '15',
                'Shpenzimet e tatimit në fitim/ Income tax expense/\nIzdvajanja za porez na dohodak': '15',
                'Tatimi në fitim/ Income tax/\nPorez na dohodak': '18',
                'Fitimi/(humbja) neto': '16',
                'Fitimi/(humbja) neto/ Net profit/(loss)/ Neto profit /(gubitak)': '16',
                'Të ardhurat tjera gjithëpërfshirëse': '17',
                'Të ardhurat tjera gjithëpërfshirëse/ Other comprehensive income/ Drugi sveobuhvatni prihod': '17',
                'Gjithsej të ardhurat gjithëpërfshirëse': '19',
                'Gjithsej të ardhurat gjithëpërfshirëse/ Total comprehensive income/(loss)/ Ukupni sveobuhvatni prihod/(gubitak)': '19',
            }
            
            full_is['CATEGORY_CODE'] = full_is['INCOME_CATEGORY'].map(is_map)
            full_is['BANK_ID'] = 11  # Credins Bank ID
            full_is['STATEMENT_TYPE'] = 'INCOME_STATEMENT'
            full_is['DT_REPORT'] = pd.to_datetime(full_is['DT_REPORT'], format='%d/%m/%Y', errors='coerce')
        
        # Create unified DataFrame
        unified_dfs = []
        
        if not full_is.empty:
            is_unified = full_is.rename(columns={
                'INCOME_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_INCOME_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_INCOME_VALUE': 'CURRENT_VALUE'
            })
            is_unified['CATEGORY_TYPE'] = 'INCOME'
            unified_dfs.append(is_unified)
        
        if not full_bs.empty:
            bs_unified = full_bs.rename(columns={
                'BALANCE_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_BALANCE_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_BALANCE_VALUE': 'CURRENT_VALUE'
            })
            bs_unified['CATEGORY_TYPE'] = 'BALANCE'
            unified_dfs.append(bs_unified)
        
        if unified_dfs:
            final_df = pd.concat(unified_dfs, ignore_index=True)
            final_df['CURR_ID'] = 1
            final_df['DATA_SOURCE'] = 'Credins Bank'
            final_df['EXTRACTION_DATE'] = datetime.now().strftime('%Y-%m-%d')
            final_df['REPORT_DATE'] = final_df['DT_REPORT'].dt.strftime('%Y-%m-%d')
            
            # Drop rows with invalid dates or categories
            final_df = final_df.dropna(subset=['DT_REPORT', 'CATEGORY_CODE'])
            
            # Sort
            final_df = final_df.sort_values(['DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_CODE'])
            
            # Reorder columns
            column_order = ['BANK_ID', 'REPORT_DATE', 'DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_TYPE',
                          'CATEGORY_CODE', 'ORIGINAL_CATEGORY', 'PREVIOUS_VALUE', 'CURRENT_VALUE',
                          'CURR_ID', 'FILE_NAME', 'DATA_SOURCE', 'EXTRACTION_DATE']
            
            final_columns = [col for col in column_order if col in final_df.columns]
            final_df = final_df[final_columns]
            
            # Save outputs
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            
            # Save unified DataFrame
            unified_path = os.path.join(OUTPUT_DIR, f"cre_unified_financial_data_{timestamp}.csv")
            final_df.to_csv(unified_path, index=False)
            print(f"\n💾 Unified data saved to: {unified_path}")
            
            # Save individual files
            if not full_is.empty:
                is_path = os.path.join(OUTPUT_DIR, f"cre_income_statement_{timestamp}.csv")
                full_is.to_csv(is_path, index=False)
            
            if not full_bs.empty:
                bs_path = os.path.join(OUTPUT_DIR, f"cre_balance_sheet_{timestamp}.csv")
                full_bs.to_csv(bs_path, index=False)
            
            # Save Excel with multiple sheets
            excel_path = os.path.join(OUTPUT_DIR, f"cre_financial_report_{timestamp}.xlsx")
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                final_df.to_excel(writer, sheet_name='Unified_Data', index=False)
                if not full_is.empty:
                    full_is.to_excel(writer, sheet_name='Income_Statement', index=False)
                if not full_bs.empty:
                    full_bs.to_excel(writer, sheet_name='Balance_Sheet', index=False)
            
            print(f"💾 Excel report saved to: {excel_path}")
            
            print(f"\n📊 UNIFIED DATAFRAME CREATED:")
            print(f"   - Total rows: {len(final_df)}")
            print(f"   - Income Statement rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'INCOME_STATEMENT'])}")
            print(f"   - Balance Sheet rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'BALANCE_SHEET'])}")
            print(f"   - Unique report dates: {final_df['REPORT_DATE'].nunique()}")
            print(f"   - Date range: {final_df['REPORT_DATE'].min()} to {final_df['REPORT_DATE'].max()}")
            
            elapsed_time = time.time() - start
            print(f"\n⏱️ Processing completed in {elapsed_time:.2f} seconds")
            
            return final_df, full_is, full_bs
        
        return pd.DataFrame(), full_is, full_bs
    
    return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

# Run the extraction
print("🚀 Starting Credins Bank financial data extraction...")
final_df, income_df, balance_df = main(force=True, extract_only=False)

if not final_df.empty:
    print("\n" + "="*60)
    print("📊 FINAL UNIFIED DATAFRAME")
    print("="*60)
    print(f"Shape: {final_df.shape}")
    print("\nFirst 10 rows:")
    print(final_df.head(10))
    
    print("\n📊 Summary by Statement Type:")
    print(final_df['STATEMENT_TYPE'].value_counts())
    
    print("\n📅 Reports by Date:")
    print(final_df['REPORT_DATE'].value_counts().sort_index())
    
    print("\n📊 Data Types:")
    print(final_df.dtypes)
else:
    print("❌ No data in final DataFrame")

📁 Created directory: cre_financial_data/current/balance-sheet
📁 Created directory: cre_financial_data/current/income-statement
📁 Created directory: cre_financial_data
📁 Created directory: cre_financial_data/original
📁 Created directory: cre_financial_data/output
🔗 Target URL: https://bankacredins-ks.com/kush-jemi/raporte-dhe-njoftime/raporte-vjetore/
🚀 Starting Credins Bank financial data extraction...

🏦 CREDINS BANK FINANCIAL DATA EXTRACTION

🌐 DOWNLOAD MODE: Downloading and processing new files
⚠️ Force mode enabled: all files will be downloaded
🔍 Scraping file links from: https://bankacredins-ks.com/kush-jemi/raporte-dhe-njoftime/raporte-vjetore/
🔍 Attempting to connect to: https://bankacredins-ks.com/kush-jemi/raporte-dhe-njoftime/raporte-vjetore/
✅ Successfully connected! Status code: 200
   📄 Found file: Pasqyrat-31.12.2025-per-tu-publikuar-ne-web-1.pdf
   📄 Found file: Pasqyrat-30.09.2025-per-tu-publikuar-ne-web.pdf
   📄 Found file: Pasqyrat-30.06.2025-per-tu-publikuar-ne-web.p

2026-03-15T19:11:08 - INFO - Processing page-1
2026-03-15 19:11:08,128 - INFO - Processing page-1
2026-03-15T19:11:08 - INFO - Processing page-3
2026-03-15 19:11:08,166 - INFO - Processing page-3



💾 File metadata saved to cre_financial_data/output/file_metadata_20260315_191108.csv

✅ Downloaded 20 new files.

📂 Processing balance sheet files...

   🔍 Processing: cre_bs_30.09.2021.xlsx
      📊 Reading Excel: cre_bs_30.09.2021.xlsx
      ✅ Extracted: 18 balance rows, 15 income rows
      ✅ Added 18 balance rows for 2021-09-30
      ✅ Added 15 income rows for 2021-09-30

   🔍 Processing: cre_bs_30.06.2025.pdf
      📊 Reading PDF: cre_bs_30.06.2025.pdf
      ❌ Error reading PDF: Ghostscript is not installed. You can install it using the instructions here: https://camelot-py.readthedocs.io/en/master/user/install-deps.html

   🔍 Processing: cre_bs_31.12.2025.pdf
      📊 Reading PDF: cre_bs_31.12.2025.pdf


2026-03-15T19:11:08 - INFO - Processing page-1
2026-03-15 19:11:08,236 - INFO - Processing page-1
2026-03-15T19:11:08 - INFO - Processing page-3
2026-03-15 19:11:08,276 - INFO - Processing page-3
2026-03-15T19:11:08 - INFO - Processing page-1
2026-03-15 19:11:08,348 - INFO - Processing page-1
2026-03-15T19:11:08 - INFO - Processing page-3
2026-03-15 19:11:08,423 - INFO - Processing page-3


      ❌ Error reading PDF: Ghostscript is not installed. You can install it using the instructions here: https://camelot-py.readthedocs.io/en/master/user/install-deps.html

   🔍 Processing: cre_bs_31.12.2024.pdf
      📊 Reading PDF: cre_bs_31.12.2024.pdf
      ❌ Error reading PDF: Ghostscript is not installed. You can install it using the instructions here: https://camelot-py.readthedocs.io/en/master/user/install-deps.html

   🔍 Processing: cre_bs_30.06.2024.xlsx
      📊 Reading Excel: cre_bs_30.06.2024.xlsx
      ✅ Extracted: 20 balance rows, 15 income rows
      ✅ Added 20 balance rows for 2024-06-30
      ✅ Added 15 income rows for 2024-06-30

   🔍 Processing: cre_bs_31.12.2022.xlsx
      📊 Reading Excel: cre_bs_31.12.2022.xlsx
      ✅ Extracted: 20 balance rows, 16 income rows
      ✅ Added 20 balance rows for 2022-12-31
      ✅ Added 16 income rows for 2022-12-31

   🔍 Processing: cre_bs_31.03.2021.xlsx
      📊 Reading Excel: cre_bs_31.03.2021.xlsx
      ✅ Extracted: 15 balance ro

2026-03-15T19:11:08 - INFO - Processing page-1
2026-03-15 19:11:08,539 - INFO - Processing page-1
2026-03-15T19:11:08 - INFO - Processing page-3
2026-03-15 19:11:08,611 - INFO - Processing page-3
2026-03-15T19:11:08 - INFO - Processing page-1
2026-03-15 19:11:08,755 - INFO - Processing page-1
2026-03-15T19:11:08 - INFO - Processing page-3
2026-03-15 19:11:08,807 - INFO - Processing page-3


      ❌ Error reading PDF: Ghostscript is not installed. You can install it using the instructions here: https://camelot-py.readthedocs.io/en/master/user/install-deps.html

   🔍 Processing: cre_bs_30.06.2022.xlsx
      📊 Reading Excel: cre_bs_30.06.2022.xlsx
      ✅ Extracted: 20 balance rows, 16 income rows
      ✅ Added 20 balance rows for 2022-06-30
      ✅ Added 16 income rows for 2022-06-30

   🔍 Processing: cre_bs_31.12.2023.xlsx
      📊 Reading Excel: cre_bs_31.12.2023.xlsx
      ✅ Extracted: 21 balance rows, 17 income rows
      ✅ Added 21 balance rows for 2023-12-31
      ✅ Added 17 income rows for 2023-12-31

   🔍 Processing: cre_bs_30.06.2021.xlsx
      📊 Reading Excel: cre_bs_30.06.2021.xlsx
      ✅ Extracted: 18 balance rows, 15 income rows
      ✅ Added 18 balance rows for 2021-06-30
      ✅ Added 15 income rows for 2021-06-30

   🔍 Processing: cre_bs_31.03.2023.xlsx
      📊 Reading Excel: cre_bs_31.03.2023.xlsx
      ✅ Extracted: 20 balance rows, 16 income rows
      ✅ A

2026-03-15T19:11:08 - INFO - Processing page-1
2026-03-15 19:11:08,892 - INFO - Processing page-1
2026-03-15T19:11:08 - INFO - Processing page-3
2026-03-15 19:11:08,931 - INFO - Processing page-3


      ✅ Extracted: 20 balance rows, 16 income rows
      ✅ Added 20 balance rows for 2023-09-30
      ✅ Added 16 income rows for 2023-09-30

   🔍 Processing: cre_bs_30.09.2025.pdf
      📊 Reading PDF: cre_bs_30.09.2025.pdf
      ❌ Error reading PDF: Ghostscript is not installed. You can install it using the instructions here: https://camelot-py.readthedocs.io/en/master/user/install-deps.html

   🔍 Processing: cre_bs_31.03.2024.xlsx
      📊 Reading Excel: cre_bs_31.03.2024.xlsx
      ✅ Extracted: 20 balance rows, 15 income rows
      ✅ Added 20 balance rows for 2024-03-31
      ✅ Added 15 income rows for 2024-03-31

   🔍 Processing: cre_bs_30.09.2022.xlsx
      📊 Reading Excel: cre_bs_30.09.2022.xlsx
      ✅ Extracted: 20 balance rows, 16 income rows
      ✅ Added 20 balance rows for 2022-09-30
      ✅ Added 16 income rows for 2022-09-30

📊 Creating DataFrames...

💾 Unified data saved to: cre_financial_data/output/cre_unified_financial_data_20260315_191108.csv
💾 Excel report saved to: c

In [1]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import re
from datetime import datetime
import openpyxl
import logging
import shutil
import tempfile
from PyPDF2 import PdfReader
import pdfplumber  # <-- Use pdfplumber instead of camelot
from collections import defaultdict

# Configure logging and start timer
start = time.time()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
pd.set_option('display.max_rows', 500, 'display.max_columns', 500, 'display.max_colwidth', 1000)

# Setup directories
BASE_DIR = "cre_financial_data"
ORIGINAL_DIR = os.path.join(BASE_DIR, "original")
CURRENT_DIR = os.path.join(BASE_DIR, "current")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Create subdirectories for categorized files
for subfolder in ['balance-sheet', 'income-statement']:
    os.makedirs(os.path.join(CURRENT_DIR, subfolder), exist_ok=True)
    print(f"📁 Created directory: {os.path.join(CURRENT_DIR, subfolder)}")

for dir_path in [BASE_DIR, ORIGINAL_DIR, OUTPUT_DIR]:
    os.makedirs(dir_path, exist_ok=True)
    print(f"📁 Created directory: {dir_path}")

# Log file for tracking processed files
PROCESSED_FILES_LOG = os.path.join(BASE_DIR, "processed_files.csv")

bank_url = 'https://bankacredins-ks.com/kush-jemi/raporte-dhe-njoftime/raporte-vjetore/'
print(f"🔗 Target URL: {bank_url}")

def load_processed_files():
    """Load list of already processed files from CSV"""
    if os.path.exists(PROCESSED_FILES_LOG):
        df = pd.read_csv(PROCESSED_FILES_LOG)
        return set(df['file_name'].tolist())
    return set()

def save_processed_file(file_name, status=1, reporting_date=None, current_name=None, is_corrupt=0):
    """Save processed file to log"""
    processed = load_processed_files()
    if file_name not in processed:
        df = pd.DataFrame({
            'file_name': [file_name],
            'processed_date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            'status': [status],
            'reporting_date': [reporting_date],
            'current_name': [current_name],
            'is_corrupt': [is_corrupt]
        })
        if os.path.exists(PROCESSED_FILES_LOG):
            existing = pd.read_csv(PROCESSED_FILES_LOG)
            df = pd.concat([existing, df], ignore_index=True)
        df.to_csv(PROCESSED_FILES_LOG, index=False)

def scrape_pdf_links(url):
    """Scrape PDF and Excel links from the Credins page"""
    print(f"🔍 Attempting to connect to: {url}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        print(f"✅ Successfully connected! Status code: {response.status_code}")
        
        soup = BeautifulSoup(response.text, 'html.parser')
        file_links = []
        
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.lower().endswith('.xlsx') or (href.lower().endswith('.pdf') and 'web' in href.lower()):
                full_url = urljoin(url, href)
                file_links.append(full_url)
                print(f"   📄 Found file: {os.path.basename(full_url)}")
        
        print(f"\n✅ Total files found: {len(file_links)}")
        return file_links
    except requests.exceptions.RequestException as e:
        print(f"❌ Error connecting to website: {e}")
        return []

def download_file(url, local_dir):
    """Download file (PDF or Excel) to local directory"""
    file_name = os.path.basename(url)
    local_path = os.path.join(local_dir, file_name)
    
    print(f"   ⬇️ Downloading: {file_name}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open(local_path, 'wb') as f:
            f.write(response.content)
        
        file_size = os.path.getsize(local_path) / 1024
        print(f"   ✅ Downloaded: {file_name} ({file_size:.1f} KB)")
        return local_path
    except Exception as e:
        print(f"   ❌ Failed to download {file_name}: {e}")
        return None

def extract_date_from_filename(filename):
    """Extract date from filename like 31.12.2023"""
    match = re.search(r'\d{2}\.\d{2}\.\d{4}', filename)
    if match:
        return match.group()
    return None

def extract_date_from_text(text):
    """Extract date from text content"""
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', text)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    return None

def get_file_creation_date(file_path, file_extension):
    """Extract creation date from file metadata"""
    dt = None
    try:
        if file_extension == 'xlsx':
            wb = openpyxl.load_workbook(file_path)
            properties = wb.properties
            created = properties.created
            modified = properties.modified
            created_date = created.date() if created else None
            modified_date = modified.date() if modified else None
            dt = created_date or modified_date
        elif file_extension == 'pdf':
            reader = PdfReader(file_path)
            metadata = reader.metadata
            
            creation_date_raw = metadata.get('/CreationDate', None)
            modified_date_raw = metadata.get('/ModDate', None)
            
            if modified_date_raw and modified_date_raw.startswith("D:"):
                modified_date_raw = modified_date_raw[2:].split('+')[0].split('-')[0]
                modified_date = datetime.strptime(modified_date_raw, "%Y%m%d%H%M%S")
                dt = modified_date
            elif creation_date_raw and creation_date_raw.startswith("D:"):
                creation_date_raw = creation_date_raw[2:].split('+')[0].split('-')[0]
                creation_date = datetime.strptime(creation_date_raw, "%Y%m%d%H%M%S")
                dt = creation_date
    except Exception as e:
        print(f"   ⚠️ Could not read metadata: {e}")
    
    return dt

def download_all(force=False):
    """Download files and categorize them locally"""
    processed = set() if force else load_processed_files()
    if force:
        print("⚠️ Force mode enabled: all files will be downloaded")
    
    print(f"🔍 Scraping file links from: {bank_url}")
    file_links = scrape_pdf_links(bank_url)
    
    if not file_links:
        print("❌ No files found")
        return []
    
    new_files = []
    file_metadata = []

    print(f"\n📥 Starting download of {len(file_links)} files...")
    
    for i, file_url in enumerate(file_links, 1):
        file_name = os.path.basename(file_url)
        print(f"\n[{i}/{len(file_links)}] Processing: {file_name}")
        
        # Skip if already processed
        if file_name in processed and not force:
            print(f"   ⏭️ File already processed. Skipping...")
            continue

        new_files.append(file_name)
        
        # Download to original directory
        file_path = download_file(file_url, ORIGINAL_DIR)
        if not file_path:
            continue

        file_extension = file_name.split('.')[-1].lower()
        
        # Get file creation date from metadata
        dt = get_file_creation_date(file_path, file_extension)
        if dt:
            print(f"   📅 File creation date: {dt}")

        # Extract date from filename
        date = extract_date_from_filename(file_name)
        
        if date:
            print(f"   📅 Extracted reporting date: {date}")
        else:
            print(f"   ⚠️ Could not extract date from filename")
            date = datetime.now().strftime('%d.%m.%Y')

        # For Credins, we create both BS and IS from the same file
        if file_extension == 'xlsx':
            bs_new_name = f'cre_bs_{date}.xlsx'
            is_new_name = f'cre_is_{date}.xlsx'
        else:  # pdf
            bs_new_name = f'cre_bs_{date}.pdf'
            is_new_name = f'cre_is_{date}.pdf'
        
        # Copy to both categorized folders
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'balance-sheet', bs_new_name))
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'income-statement', is_new_name))
        
        file_metadata.append({
            'file_name': file_name,
            'file_path': file_path,
            'bs_current_name': bs_new_name,
            'is_current_name': is_new_name,
            'file_extension': file_extension,
            'reporting_date': date,
            'creation_date': dt,
            'download_date': datetime.now(),
            'is_corrupt': 0
        })
        
        print(f"   ✅ Created: {bs_new_name} and {is_new_name}")
        
        # Mark as processed
        save_processed_file(file_name, reporting_date=date, current_name=f"{bs_new_name}|{is_new_name}")

    # Save metadata
    if file_metadata:
        metadata_df = pd.DataFrame(file_metadata)
        metadata_path = os.path.join(OUTPUT_DIR, f"file_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        metadata_df.to_csv(metadata_path, index=False)
        print(f"\n💾 File metadata saved to {metadata_path}")

    if new_files:
        print(f"\n✅ Downloaded {len(new_files)} new files.")
    else:
        print("\n✅ No new files detected.")

    # Return list of all new current files
    current_files = []
    for m in file_metadata:
        current_files.append(m['bs_current_name'])
        current_files.append(m['is_current_name'])
    return current_files

def replace_negatives(value):
    """Convert parenthesized numbers to negative"""
    if pd.isna(value) or str(value).strip() in ['', '-', '0']:
        return '0'
    val = str(value).strip()
    if '(' in val and ')' in val:
        num = val.replace('(', '').replace(')', '').replace(',', '').strip()
        return f"-{num}"
    return val

def extract_table_from_pdf(pdf_path, page_num):
    """Extract table from PDF using pdfplumber"""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            if len(pdf.pages) > page_num:
                page = pdf.pages[page_num]
                tables = page.extract_tables()
                
                if tables:
                    # Take the first table
                    table = tables[0]
                    df = pd.DataFrame(table)
                    return df
    except Exception as e:
        print(f"      ⚠️ Error extracting table: {e}")
    
    return pd.DataFrame()

def read_cre_pdf(pdf_path, pdf_filename):
    """Read Credins PDF file using pdfplumber (no Ghostscript needed)"""
    try:
        print(f"      📊 Reading PDF with pdfplumber: {pdf_filename}")
        
        # Try to get date from PDF text
        date = None
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text() or ""
            date = extract_date_from_text(text)
        
        if not date:
            date = extract_date_from_filename(pdf_filename)
        
        # Extract balance sheet from page 1
        bal_df = extract_table_from_pdf(pdf_path, 0)
        
        # Extract income statement from page 3
        inc_df = extract_table_from_pdf(pdf_path, 2)
        
        if bal_df.empty or inc_df.empty:
            print(f"      ⚠️ Could not extract tables, trying alternative method...")
            # Try alternative: read all text and parse manually
            with pdfplumber.open(pdf_path) as pdf:
                full_text = ""
                for page in pdf.pages:
                    full_text += page.extract_text() + "\n"
                
                # Simple parsing logic - you may need to adjust based on actual PDF format
                lines = full_text.split('\n')
                bal_data = []
                inc_data = []
                
                in_balance = False
                in_income = False
                
                for line in lines:
                    line = line.strip()
                    if not line:
                        continue
                    
                    if 'BILANCI' in line.upper() or 'BALANCE' in line.upper():
                        in_balance = True
                        in_income = False
                        continue
                    elif 'PASQYRA' in line.upper() or 'INCOME' in line.upper():
                        in_balance = False
                        in_income = True
                        continue
                    
                    values = re.findall(r'\(?[-\d,]+(?:\.\d+)?\)?', line)
                    if len(values) >= 2:
                        category = re.sub(r'\(?[-\d,]+(?:\.\d+)?\)?', '', line).strip()
                        if category and in_balance:
                            bal_data.append([category, values[0], values[1] if len(values) > 1 else '0'])
                        elif category and in_income:
                            inc_data.append([category, values[0], values[1] if len(values) > 1 else '0'])
                
                if bal_data:
                    bal_df = pd.DataFrame(bal_data)
                if inc_data:
                    inc_df = pd.DataFrame(inc_data)
        
        if bal_df.empty:
            print(f"      ⚠️ No balance sheet data extracted")
            bal_df = pd.DataFrame(columns=[0, 1, 2])
        
        if inc_df.empty:
            print(f"      ⚠️ No income statement data extracted")
            inc_df = pd.DataFrame(columns=[0, 1, 2])
        
        # Ensure we have at least 3 columns
        while bal_df.shape[1] < 3:
            bal_df[bal_df.shape[1]] = ''
        while inc_df.shape[1] < 3:
            inc_df[inc_df.shape[1]] = ''
        
        # Clean data
        bal_df = bal_df.iloc[:, :3]
        inc_df = inc_df.iloc[:, :3]
        
        # Remove header rows that might be text
        bal_df = bal_df[~bal_df[0].astype(str).str.contains('BILANCI|BALANCE|PASURIT|DETYRIM', case=False, na=False)]
        inc_df = inc_df[~inc_df[0].astype(str).str.contains('PASQYRA|INCOME|TË HYRAT|SHPENZIMET', case=False, na=False)]
        
        # Clean category names
        bal_df[0] = bal_df[0].astype(str).str.replace('/ .*', '', regex=True).str.strip()
        inc_df[0] = inc_df[0].astype(str).str.replace('/ .*', '', regex=True).str.replace('\n.*', '', regex=True).str.strip()
        
        # Remove empty rows
        bal_df = bal_df[(bal_df[0] != '') & (bal_df[1] != '') & (bal_df[2] != '')]
        inc_df = inc_df[(inc_df[0] != '') & (inc_df[1] != '') & (inc_df[2] != '')]
        
        # Convert to numeric
        for col in [1, 2]:
            bal_df[col] = pd.to_numeric(bal_df[col].astype(str).str.replace('-', '0').str.replace(',', '').apply(replace_negatives), errors='coerce').fillna(0)
            inc_df[col] = pd.to_numeric(inc_df[col].astype(str).str.replace('-', '0').str.replace(',', '').apply(replace_negatives), errors='coerce').fillna(0)
        
        # Rename columns
        bal_df.columns = ['BALANCE_CATEGORY', 'CURRENT_BALANCE_VALUE', 'PREVIOUS_BALANCE_VALUE']
        inc_df.columns = ['INCOME_CATEGORY', 'CURRENT_INCOME_VALUE', 'PREVIOUS_INCOME_VALUE']
        
        # Add metadata
        if date:
            date = date.replace('.', '/')
        
        bal_df['DT_REPORT'] = date
        inc_df['DT_REPORT'] = date
        bal_df['FILE_NAME'] = pdf_filename
        inc_df['FILE_NAME'] = pdf_filename.replace('bs', 'is')
        
        print(f"      ✅ Extracted: {len(bal_df)} balance rows, {len(inc_df)} income rows")
        return bal_df, inc_df
    
    except Exception as e:
        print(f"      ❌ Error reading PDF: {e}")
        return pd.DataFrame(), pd.DataFrame()

def read_cre_excel(excel_path, excel_filename):
    """Read Credins Excel file (BS and IS sheets)"""
    try:
        print(f"      📊 Reading Excel: {excel_filename}")
        
        # Read balance sheet sheet (default)
        df = pd.read_excel(excel_path)
        df = df.dropna(subset=['Unnamed: 1'])
        df.drop(['Unnamed: 0', 'Unnamed: 2'], axis=1, inplace=True)
        
        # Handle column rearrangement if needed
        try:
            df['Unnamed: 4'] = df['Unnamed: 5']
            df.drop(['Unnamed: 5'], axis=1, inplace=True)
        except:
            pass
        
        # Clean numeric values
        df['Unnamed: 3'] = df['Unnamed: 3'].apply(lambda x: round(x, 0) if pd.notna(x) else 0)
        df['Unnamed: 4'] = df['Unnamed: 4'].apply(lambda x: round(x, 0) if pd.notna(x) else 0)
        df['Unnamed: 3'] = pd.to_numeric(df['Unnamed: 3'], errors='coerce').fillna(0)
        df['Unnamed: 4'] = pd.to_numeric(df['Unnamed: 4'], errors='coerce').fillna(0)
        df = df.dropna(subset=['Unnamed: 3', 'Unnamed: 4'], how='all')
        df.fillna(0, inplace=True)
        
        # Read income statement sheet
        df2 = pd.read_excel(excel_path, sheet_name='IS')
        df2 = df2.dropna(subset=['Unnamed: 1'])
        df2.drop(['Unnamed: 0', 'Unnamed: 2'], axis=1, inplace=True)
        
        # Handle column rearrangement if needed
        try:
            df2['Unnamed: 4'] = df2['Unnamed: 5']
            df2.drop(['Unnamed: 5'], axis=1, inplace=True)
        except:
            pass
        
        # Clean numeric values
        df2['Unnamed: 3'] = df2['Unnamed: 3'].apply(lambda x: round(x, 0) if pd.notna(x) else 0)
        df2['Unnamed: 4'] = df2['Unnamed: 4'].apply(lambda x: round(x, 0) if pd.notna(x) else 0)
        df2['Unnamed: 3'] = pd.to_numeric(df2['Unnamed: 3'], errors='coerce').fillna(0)
        df2['Unnamed: 4'] = pd.to_numeric(df2['Unnamed: 4'], errors='coerce').fillna(0)
        df2 = df2.dropna(subset=['Unnamed: 3', 'Unnamed: 4'], how='all')
        df2.fillna(0, inplace=True)
        
        # Set column names
        df.columns = ['BALANCE_CATEGORY', 'CURRENT_BALANCE_VALUE', 'PREVIOUS_BALANCE_VALUE']
        df2.columns = ['INCOME_CATEGORY', 'CURRENT_INCOME_VALUE', 'PREVIOUS_INCOME_VALUE']
        
        # Extract date from filename
        date = extract_date_from_filename(excel_filename)
        if date:
            date = date.replace('.', '/')
        
        df['DT_REPORT'] = date
        df2['DT_REPORT'] = date
        df['FILE_NAME'] = excel_filename
        df2['FILE_NAME'] = excel_filename.replace('bs', 'is')
        
        print(f"      ✅ Extracted: {len(df)} balance rows, {len(df2)} income rows")
        return df, df2
    
    except Exception as e:
        print(f"      ❌ Error reading Excel: {e}")
        return pd.DataFrame(), pd.DataFrame()

# Data storage
income_data = defaultdict(list)
balance_data = defaultdict(list)

def process_cre_files():
    """Process all Credins files and populate data dictionaries"""
    
    # Process balance sheet files
    bs_dir = os.path.join(CURRENT_DIR, 'balance-sheet')
    if os.path.exists(bs_dir):
        print(f"\n📂 Processing balance sheet files...")
        for filename in os.listdir(bs_dir):
            if filename.startswith('cre_bs_'):
                file_path = os.path.join(bs_dir, filename)
                print(f"\n   🔍 Processing: {filename}")
                
                # Extract date from filename
                date_match = re.search(r'\d{2}\.\d{2}\.\d{4}', filename)
                dt_report = date_match.group() if date_match else None
                dt_report_formatted = datetime.strptime(dt_report, '%d.%m.%Y').strftime('%Y-%m-%d') if dt_report else None
                
                if not dt_report:
                    print(f"      ⚠️ Could not extract date from filename")
                    continue
                
                file_extension = filename.split('.')[-1].lower()
                
                # Get corresponding income statement filename
                is_filename = filename.replace('bs', 'is')
                is_path = os.path.join(CURRENT_DIR, 'income-statement', is_filename)
                
                if os.path.exists(is_path):
                    # Read based on file type
                    if file_extension == 'pdf':
                        balance_df, income_df = read_cre_pdf(file_path, filename)
                    else:  # xlsx
                        balance_df, income_df = read_cre_excel(file_path, filename)
                    
                    # Add to balance data
                    if not balance_df.empty and dt_report_formatted:
                        for _, row in balance_df.iterrows():
                            balance_data[dt_report_formatted].append({
                                'Category': row['BALANCE_CATEGORY'],
                                'PREVIOUS_QUARTER': row['PREVIOUS_BALANCE_VALUE'],
                                'CURRENT_QUARTER': row['CURRENT_BALANCE_VALUE'],
                                'DT_REPORT': dt_report_formatted,
                                'FILE_NAME': filename
                            })
                        print(f"      ✅ Added {len(balance_df)} balance rows for {dt_report_formatted}")
                    
                    # Add to income data
                    if not income_df.empty and dt_report_formatted:
                        for _, row in income_df.iterrows():
                            income_data[dt_report_formatted].append({
                                'Category': row['INCOME_CATEGORY'],
                                'PREVIOUS_QUARTER': row['PREVIOUS_INCOME_VALUE'],
                                'CURRENT_QUARTER': row['CURRENT_INCOME_VALUE'],
                                'DT_REPORT': dt_report_formatted,
                                'FILE_NAME': is_filename
                            })
                        print(f"      ✅ Added {len(income_df)} income rows for {dt_report_formatted}")

def clean_numeric_values(df, prev_col, curr_col):
    """Clean and convert numeric columns"""
    for col in [prev_col, curr_col]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

def main(force=False, extract_only=False):
    """Main execution function that returns unified DataFrames"""
    
    print("\n" + "="*60)
    print("🏦 CREDINS BANK FINANCIAL DATA EXTRACTION")
    print("="*60)
    
    # Clear previous data
    global income_data, balance_data
    income_data.clear()
    balance_data.clear()
    
    # Decide which files to process
    if extract_only:
        print("\n📂 EXTRACT-ONLY MODE: Processing existing files")
        new_files = []
        for category in ['balance-sheet', 'income-statement']:
            category_dir = os.path.join(CURRENT_DIR, category)
            if os.path.exists(category_dir):
                category_files = [f for f in os.listdir(category_dir) if f.startswith('cre_') and f.endswith(('.pdf', '.xlsx'))]
                new_files.extend(category_files)
        print(f"Found {len(new_files)} existing files")
    else:
        print("\n🌐 DOWNLOAD MODE: Downloading and processing new files")
        new_files = download_all(force=force)
    
    if new_files or extract_only:
        # Process all Credins files
        process_cre_files()
        
        # Create DataFrames
        print("\n📊 Creating DataFrames...")
        
        # Balance Sheet DataFrame
        full_bs = pd.DataFrame()
        for date, data in balance_data.items():
            df = pd.DataFrame(data)
            full_bs = pd.concat([full_bs, df], ignore_index=True)
        
        if not full_bs.empty:
            full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_bs = clean_numeric_values(full_bs, 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE')
            
            # Category mapping for balance sheet
            bs_map = {
                'Paraja e gatshme dhe gjendja me BQK-në': '20',
                'Paraja e gatshme dhe gjendja me BQK-në/ Cash and balances with CBK/ Bilans gotovine sa CBK': '20',
                'Kërkesat ndaj bankave': '21',
                'Kërkesat ndaj bankave/ Claims on banks/ Potraživanja bankama': '21',
                'Bonot e thesarit': '22',
                'Investimet në letra me vlerë': '23',
                'Investimet në letra me vlerë/ Investment Securities/ Hartije od vrednosti': '23',
                'Kreditë dhe paradhëniet ndaj klientëve': '26',
                'Kreditë dhe paradhëniet ndaj klientëve/ Loans and advances to customers/ Plasmani klijentima': '26',
                'Patundshmëritë dhe pajisjet': '27',
                'Patundshmëritë dhe pajisjet/ Property and equipment/ Imovina I oprema': '27',
                'Pasuritë e paprekshme': '28',
                'Pasuritë e paprekshme/ Intangible assets/ Nematerijalna aktiva': '28',
                'Pasuritë tatimore të shtyra': '29',
                'Pasuritë tatimore të shtyra/ Deferred tax assets/ Odložena poreska sredstva': '29',
                'Pasuritë tjera': '30',
                'Pasuritë tjera/ Other assets/ Ostala aktiva': '30',
                'Gjithsej pasuritë': '31',
                'Gjithsej pasuritë/ Total assets/ Ukupna aktiva': '31',
                'Depozitat e klientëve': '32',
                'Depozitat e klientëve/ Customer deposits/ Depoziti klijenata': '32',
                'Detyrimet ndaj bankave': '33',
                'Detyrimet ndaj bankave/ Due to banks/ Obaveze prema banci': '33',
                'Fondet tjera të huazuara': '34',
                'Detyrimet tatimore të shtyra': '35',
                'Detyrimet tatimore të shtyra/ Deferred tax liabilities/ Odlozene poreske obaveze': '35',
                'Detyrimet tjera': '36',
                'Detyrimet tjera/ Other liabilities/ Druge obaveze ': '36',
                'Gjithsej detyrimet': '37',
                'Gjithsej detyrimet/ Total liabilities/ Ukupne obaveze ': '37',
                'Kapitali aksionar': '38',
                'Kapitali aksionar/ Share capital/ Deoničarske akcize': '38',
                'Primi I aksioneve': '39',
                'Primi I aksioneve/ Share premium/ Akcionarska premija': '39',
                'Rezervat e kapitalit': '40',
                'Fitimi i mbajtur/(humbja) nga vitet paraprake': '41',
                'Fitimi i mbajtur/(humbja) nga vitet paraprake/ Retained profit/loss from previous years/ Nerasporedena dobit/gubitak iz prethodnih godina': '41',
                'Fitimi/(humbja) e vitit aktual': '42',
                'Fitimi/(humbja) e vitit aktual/ current year profit/loss/ gubitak iz prethodnih godina': '42',
                'Përbërësit tjerë të ekuitetit': '43',
                'Përbërësit tjerë të ekuitetit/ Other equity capital components/ Ostale komponente akcijskog kapitala ': '43',
                'Gjithsej ekuiteti i aksionarëve': '44',
                'Gjithsej ekuiteti i aksionarëve/ Total shareholder\'s equity/ Ukupni kapital deoničara': '44',
                'Gjithsej detyrimet dhe ekuiteti i aksionarëve': '45',
                'Gjithsej detyrimet dhe ekuiteti i aksionarëve/ Total liabilities and shareholder\'s equity/ Ukupne obaveze i kapital deoničara': '45',
            }
            
            full_bs['CATEGORY_CODE'] = full_bs['BALANCE_CATEGORY'].map(bs_map)
            full_bs['BANK_ID'] = 11  # Credins Bank ID
            full_bs['STATEMENT_TYPE'] = 'BALANCE_SHEET'
            full_bs['DT_REPORT'] = pd.to_datetime(full_bs['DT_REPORT'], format='%Y-%m-%d', errors='coerce')
        
        # Income Statement DataFrame
        full_is = pd.DataFrame()
        for date, data in income_data.items():
            df = pd.DataFrame(data)
            full_is = pd.concat([full_is, df], ignore_index=True)
        
        if not full_is.empty:
            full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_is = clean_numeric_values(full_is, 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE')
            
            # Category mapping for income statement
            is_map = {
                'Të hyrat nga interesi': '1',
                'Të hyrat nga interesi/ interest income/ kamatni prihod': '1',
                'Shpenzimet e interesit': '2',
                'Shpenzimet e interesit/ interest expense/ kamatni rashod': '2',
                'Neto të hyrat nga interesi': '3',
                'Neto të hyrat nga interesi/ net interest income/ neto kamatni prihod': '3',
                'Të hyrat nga tarifat dhe komisionet': '4',
                'Të hyrat nga tarifat dhe komisionet/ Fee and commission income/ Prihod od naknada i provizija': '4',
                'Shpenzimet e tarifave dhe komisioneve': '5',
                'Shpenzimet e tarifave dhe komisioneve/ Fee and commission expense/ Rashod od naknada i provizija': '5',
                'Neto të hyrat nga tarifat dhe komisionet': '6',
                'Neto të hyrat nga tarifat dhe komisionet/ Net fee and commission income/ Neto dobitak na ime naknada i provizija': '6',
                'Neto të hyrat nga tregtimi': '7',
                'Neto të hyrat nga tregtimi/ Net trading profit/ Neto profit od trgovanja': '7',
                'Neto të hyrat nga instrumentet tjera financiare': '8',
                'Neto të hyrat nga instrumentet tjera financiare/ Net income from other financial instruments/ Neto prihod iz drugih finansijskih instrumenata': '8',
                'Neto të hyrat (shpenzimet) tjera operative': '9',
                'Neto të hyrat (shpenzimet) tjera operative/ Net other operating income (expense) / Drugi neto operativni prihod (rashod)': '9',
                'Gjithsej të hyrat': '10',
                'Gjithsej të hyrat/ Total income / Ukupna dobit': '10',
                'Provizionet për humbjet nga kreditë': '11',
                'Provizionet për humbjet nga kreditë/ Impairment losses on loans/ Gubici na problematičnim kreditima': '11',
                'Provizionet tjera': '13',
                'Fitimi/(humbja) para tatimit': '14',
                'Fitimi/(humbja) para tatimit/ Profit/(loss) before taxation/Profit / (gubitak) pre oporezivanja': '14',
                'Shpenzimet e tatimit në fitim': '15',
                'Shpenzimet e tatimit në fitim/ Income tax expense/\nIzdvajanja za porez na dohodak': '15',
                'Tatimi në fitim/ Income tax/\nPorez na dohodak': '18',
                'Fitimi/(humbja) neto': '16',
                'Fitimi/(humbja) neto/ Net profit/(loss)/ Neto profit /(gubitak)': '16',
                'Të ardhurat tjera gjithëpërfshirëse': '17',
                'Të ardhurat tjera gjithëpërfshirëse/ Other comprehensive income/ Drugi sveobuhvatni prihod': '17',
                'Gjithsej të ardhurat gjithëpërfshirëse': '19',
                'Gjithsej të ardhurat gjithëpërfshirëse/ Total comprehensive income/(loss)/ Ukupni sveobuhvatni prihod/(gubitak)': '19',
            }
            
            full_is['CATEGORY_CODE'] = full_is['INCOME_CATEGORY'].map(is_map)
            full_is['BANK_ID'] = 11  # Credins Bank ID
            full_is['STATEMENT_TYPE'] = 'INCOME_STATEMENT'
            full_is['DT_REPORT'] = pd.to_datetime(full_is['DT_REPORT'], format='%Y-%m-%d', errors='coerce')
        
        # Create unified DataFrame
        unified_dfs = []
        
        if not full_is.empty:
            is_unified = full_is.rename(columns={
                'INCOME_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_INCOME_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_INCOME_VALUE': 'CURRENT_VALUE'
            })
            is_unified['CATEGORY_TYPE'] = 'INCOME'
            unified_dfs.append(is_unified)
        
        if not full_bs.empty:
            bs_unified = full_bs.rename(columns={
                'BALANCE_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_BALANCE_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_BALANCE_VALUE': 'CURRENT_VALUE'
            })
            bs_unified['CATEGORY_TYPE'] = 'BALANCE'
            unified_dfs.append(bs_unified)
        
        if unified_dfs:
            final_df = pd.concat(unified_dfs, ignore_index=True)
            final_df['CURR_ID'] = 1
            final_df['DATA_SOURCE'] = 'Credins Bank'
            final_df['EXTRACTION_DATE'] = datetime.now().strftime('%Y-%m-%d')
            final_df['REPORT_DATE'] = final_df['DT_REPORT'].dt.strftime('%Y-%m-%d')
            
            # Drop rows with invalid dates or categories
            final_df = final_df.dropna(subset=['DT_REPORT', 'CATEGORY_CODE'])
            
            # Sort
            final_df = final_df.sort_values(['DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_CODE'])
            
            # Reorder columns
            column_order = ['BANK_ID', 'REPORT_DATE', 'DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_TYPE',
                          'CATEGORY_CODE', 'ORIGINAL_CATEGORY', 'PREVIOUS_VALUE', 'CURRENT_VALUE',
                          'CURR_ID', 'FILE_NAME', 'DATA_SOURCE', 'EXTRACTION_DATE']
            
            final_columns = [col for col in column_order if col in final_df.columns]
            final_df = final_df[final_columns]
            
            # Save outputs
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            
            # Save unified DataFrame
            unified_path = os.path.join(OUTPUT_DIR, f"cre_unified_financial_data_{timestamp}.csv")
            final_df.to_csv(unified_path, index=False)
            print(f"\n💾 Unified data saved to: {unified_path}")
            
            # Save individual files
            if not full_is.empty:
                is_path = os.path.join(OUTPUT_DIR, f"cre_income_statement_{timestamp}.csv")
                full_is.to_csv(is_path, index=False)
            
            if not full_bs.empty:
                bs_path = os.path.join(OUTPUT_DIR, f"cre_balance_sheet_{timestamp}.csv")
                full_bs.to_csv(bs_path, index=False)
            
            # Save Excel with multiple sheets
            excel_path = os.path.join(OUTPUT_DIR, f"cre_financial_report_{timestamp}.xlsx")
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                final_df.to_excel(writer, sheet_name='Unified_Data', index=False)
                if not full_is.empty:
                    full_is.to_excel(writer, sheet_name='Income_Statement', index=False)
                if not full_bs.empty:
                    full_bs.to_excel(writer, sheet_name='Balance_Sheet', index=False)
            
            print(f"💾 Excel report saved to: {excel_path}")
            
            print(f"\n📊 UNIFIED DATAFRAME CREATED:")
            print(f"   - Total rows: {len(final_df)}")
            print(f"   - Income Statement rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'INCOME_STATEMENT'])}")
            print(f"   - Balance Sheet rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'BALANCE_SHEET'])}")
            print(f"   - Unique report dates: {final_df['REPORT_DATE'].nunique()}")
            print(f"   - Date range: {final_df['REPORT_DATE'].min()} to {final_df['REPORT_DATE'].max()}")
            
            elapsed_time = time.time() - start
            print(f"\n⏱️ Processing completed in {elapsed_time:.2f} seconds")
            
            return final_df, full_is, full_bs
        
        return pd.DataFrame(), full_is, full_bs
    
    return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

# Run the extraction
print("🚀 Starting Credins Bank financial data extraction...")
final_df, income_df, balance_df = main(force=True, extract_only=False)

if not final_df.empty:
    print("\n" + "="*60)
    print("📊 FINAL UNIFIED DATAFRAME")
    print("="*60)
    print(f"Shape: {final_df.shape}")
    print("\nFirst 10 rows:")
    print(final_df.head(10))
    
    print("\n📊 Summary by Statement Type:")
    print(final_df['STATEMENT_TYPE'].value_counts())
    
    print("\n📅 Reports by Date:")
    print(final_df['REPORT_DATE'].value_counts().sort_index())
    
    print("\n📊 Data Types:")
    print(final_df.dtypes)
else:
    print("❌ No data in final DataFrame")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


📁 Created directory: cre_financial_data/current/balance-sheet
📁 Created directory: cre_financial_data/current/income-statement
📁 Created directory: cre_financial_data
📁 Created directory: cre_financial_data/original
📁 Created directory: cre_financial_data/output
🔗 Target URL: https://bankacredins-ks.com/kush-jemi/raporte-dhe-njoftime/raporte-vjetore/
🚀 Starting Credins Bank financial data extraction...

🏦 CREDINS BANK FINANCIAL DATA EXTRACTION

🌐 DOWNLOAD MODE: Downloading and processing new files
⚠️ Force mode enabled: all files will be downloaded
🔍 Scraping file links from: https://bankacredins-ks.com/kush-jemi/raporte-dhe-njoftime/raporte-vjetore/
🔍 Attempting to connect to: https://bankacredins-ks.com/kush-jemi/raporte-dhe-njoftime/raporte-vjetore/
✅ Successfully connected! Status code: 200
   📄 Found file: Pasqyrat-31.12.2025-per-tu-publikuar-ne-web-1.pdf
   📄 Found file: Pasqyrat-30.09.2025-per-tu-publikuar-ne-web.pdf
   📄 Found file: Pasqyrat-30.06.2025-per-tu-publikuar-ne-web.p

In [3]:
final_df.to_csv("CRE.csv")

In [4]:
final_df

,BANK_ID,REPORT_DATE,DT_REPORT,STATEMENT_TYPE,CATEGORY_TYPE,CATEGORY_CODE,ORIGINAL_CATEGORY,PREVIOUS_VALUE,CURRENT_VALUE,CURR_ID,FILE_NAME,DATA_SOURCE,EXTRACTION_DATE
344,11,2021-03-31,2021-03-31,BALANCE_SHEET,BALANCE,20,Paraja e gatshme dhe gjendja me BQK-në,8031.0,8137.0,1,cre_bs_31.03.2021.xlsx,Credins Bank,2026-03-18
345,11,2021-03-31,2021-03-31,BALANCE_SHEET,BALANCE,21,Kërkesat ndaj bankave,0.0,0.0,1,cre_bs_31.03.2021.xlsx,Credins Bank,2026-03-18
346,11,2021-03-31,2021-03-31,BALANCE_SHEET,BALANCE,22,Bonot e thesarit,0.0,0.0,1,cre_bs_31.03.2021.xlsx,Credins Bank,2026-03-18
347,11,2021-03-31,2021-03-31,BALANCE_SHEET,BALANCE,23,Investimet në letra me vlerë,0.0,763.0,1,cre_bs_31.03.2021.xlsx,Credins Bank,2026-03-18
348,11,2021-03-31,2021-03-31,BALANCE_SHEET,BALANCE,26,Kreditë dhe paradhëniet ndaj klientëve,6.0,722.0,1,cre_bs_31.03.2021.xlsx,Credins Bank,2026-03-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
28,11,2024-06-30,2024-06-30,INCOME_STATEMENT,INCOME,5,Shpenzimet e tarifave dhe komisioneve/ Fee and commission expense/ Rashod od naknada i provizija,-32.0,-75.0,1,cre_is_30.06.2024.xlsx,Credins Bank,2026-03-18
29,11,2024-06-30,2024-06-30,INCOME_STATEMENT,INCOME,6,Neto të hyrat nga tarifat dhe komisionet/ Net fee and commission income/ Neto dobitak na ime naknada i provizija,90.0,167.0,1,cre_is_30.06.2024.xlsx,Credins Bank,2026-03-18
30,11,2024-06-30,2024-06-30,INCOME_STATEMENT,INCOME,7,Neto të hyrat nga tregtimi/ Net trading profit/ Neto profit od trgovanja,-12.0,-12.0,1,cre_is_30.06.2024.xlsx,Credins Bank,2026-03-18
31,11,2024-06-30,2024-06-30,INCOME_STATEMENT,INCOME,8,Neto të hyrat nga instrumentet tjera financiare/ Net income from other financial instruments/ Neto prihod iz drugih finansijskih instrumenata,48.0,100.0,1,cre_is_30.06.2024.xlsx,Credins Bank,2026-03-18
